# 02 — Train YOLOv8 Pothole Detector
Trains a YOLOv8n baseline on the merged pothole dataset.
Run 00_data_ingest.ipynb first and confirm data counts with 01_data_audit.ipynb before running this.

In [ ]:
import torch
from ultralytics import YOLO
from pathlib import Path
import yaml

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Verify dataset config

In [ ]:
DATA_YAML = Path("data/processed/potholes_yolo/data.yaml")
assert DATA_YAML.exists(), f"data.yaml not found at {DATA_YAML}"

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
print(yaml.dump(cfg))

base = Path(cfg["path"])
for split in ["train", "val"]:
    img_dir = base / cfg[split]
    n = len(list(img_dir.glob("*")))
    print(f"{split}: {n} images at {img_dir}")
    assert n > 0, f"No images found for split '{split}'"

## 2. Hyperparameters

In [ ]:
HYPERPARAMS = {
    "model":    "yolov8n.pt",   # change to yolov8s.pt for a stronger baseline
    "data":     str(DATA_YAML),
    "epochs":   50,
    "imgsz":    640,
    "batch":    16,             # reduce to 8 if VRAM < 8GB
    "lr0":      0.01,
    "lrf":      0.01,
    "mosaic":   1.0,
    "degrees":  5.0,
    "flipud":   0.0,
    "fliplr":   0.5,
    "project":  "ml/runs/train",
    "name":     "potholes_yolov8n_v1",
    "exist_ok": True,
    "device":   0 if torch.cuda.is_available() else "cpu",
    "workers":  4,
    "patience": 15,
    "save":     True,
    "plots":    True,
}
print(HYPERPARAMS)

## 3. Train

In [ ]:
model = YOLO(HYPERPARAMS["model"])

results = model.train(
    data=HYPERPARAMS["data"],
    epochs=HYPERPARAMS["epochs"],
    imgsz=HYPERPARAMS["imgsz"],
    batch=HYPERPARAMS["batch"],
    lr0=HYPERPARAMS["lr0"],
    lrf=HYPERPARAMS["lrf"],
    mosaic=HYPERPARAMS["mosaic"],
    degrees=HYPERPARAMS["degrees"],
    flipud=HYPERPARAMS["flipud"],
    fliplr=HYPERPARAMS["fliplr"],
    project=HYPERPARAMS["project"],
    name=HYPERPARAMS["name"],
    exist_ok=HYPERPARAMS["exist_ok"],
    device=HYPERPARAMS["device"],
    workers=HYPERPARAMS["workers"],
    patience=HYPERPARAMS["patience"],
    save=HYPERPARAMS["save"],
    plots=HYPERPARAMS["plots"],
)

RUN_DIR = Path(HYPERPARAMS["project"]) / HYPERPARAMS["name"]
print(f"Run saved to: {RUN_DIR}")

## 4. Training results

In [ ]:
from IPython.display import Image as IPImage, display

results_img = RUN_DIR / "results.png"
if results_img.exists():
    display(IPImage(str(results_img)))
else:
    print("results.png not found — check run directory")

In [ ]:
import pandas as pd

results_csv = RUN_DIR / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    best_epoch = df["metrics/mAP50(B)"].idxmax()
    best = df.loc[best_epoch]
    print(f"Best epoch:       {int(best['epoch'])}")
    print(f"mAP50:            {best['metrics/mAP50(B)']:.4f}")
    print(f"mAP50-95:         {best['metrics/mAP50-95(B)']:.4f}")
    print(f"Precision:        {best['metrics/precision(B)']:.4f}")
    print(f"Recall:           {best['metrics/recall(B)']:.4f}")
else:
    print("results.csv not found")

## 5. Validation on val split

In [ ]:
best_weights = RUN_DIR / "weights" / "best.pt"
assert best_weights.exists(), f"best.pt not found at {best_weights}"

val_model = YOLO(str(best_weights))
val_results = val_model.val(data=str(DATA_YAML), imgsz=640, device=HYPERPARAMS["device"])

## 6. Inference spot check — 5 random val images

In [ ]:
import random
from PIL import Image as PILImage
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

val_images = list((Path("data/processed/potholes_yolo/images/val")).glob("*.jpg"))
val_images += list((Path("data/processed/potholes_yolo/images/val")).glob("*.png"))
sample = random.sample(val_images, min(5, len(val_images)))

infer_model = YOLO(str(best_weights))
preds = infer_model.predict(source=[str(p) for p in sample], imgsz=640, conf=0.25, save=False)

fig, axes = plt.subplots(1, len(sample), figsize=(18, 4))
for ax, result in zip(axes, preds):
    ax.imshow(result.plot()[..., ::-1])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Export weights

In [ ]:
WEIGHTS_OUT = Path("ml/models/weights")
WEIGHTS_OUT.mkdir(parents=True, exist_ok=True)

shutil.copy2(best_weights, WEIGHTS_OUT / "potholes_yolov8n_v1_best.pt")
print(f"Weights copied to {WEIGHTS_OUT / 'potholes_yolov8n_v1_best.pt'}")

onnx_path = val_model.export(format="onnx", imgsz=640)
shutil.copy2(onnx_path, WEIGHTS_OUT / "potholes_yolov8n_v1.onnx")
print(f"ONNX exported to {WEIGHTS_OUT / 'potholes_yolov8n_v1.onnx'}")

## 8. Notes and next steps
- If mAP50 < 0.5: inspect false positives in 03_eval_detector.ipynb
- If mAP50 > 0.7: try yolov8s.pt and add HRP4K dataset
- Next: run 03_eval_detector.ipynb for error analysis
- Next: run 04_video_inference.ipynb to test on road footage